# Larp-LLM — Colab Playground

A small chat interface for trying [`celestial-sora/Larp-LLM`](https://huggingface.co/celestial-sora/Larp-LLM). The repository contains a LoRA adapter for `unsloth/gemma-3-12b-it-unsloth-bnb-4bit`; this notebook loads the adapter and serves a Gradio chat UI.

**Runtime:** In Colab choose **Runtime → Change runtime type → GPU**. A GPU with enough VRAM is required; T4 may be tight for Gemma 3 12B. First load downloads several GB and can take a while.

In [ ]:
# Install the inference and UI libraries\n# Keep Colab's gcsfs/fsspec pair on matching versions to avoid resolver conflicts.\n!pip -q install --upgrade "fsspec==2025.12.0" "gcsfs==2025.12.0" unsloth gradio\n

In [ ]:
# Load the fine-tuned adapter (public Hugging Face repo; no token needed)
import torch
from unsloth import FastModel

MODEL_ID = "celestial-sora/Larp-LLM"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastModel.for_inference(model)

print("Loaded:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


## Chat interface

Adjust the sampling controls while chatting. **Clear** starts a fresh conversation; the system prompt and chat history are included in each generation.

In [ ]:
import gradio as gr

DEFAULT_SYSTEM = "You are Larper LLM. Respond in an internet-native, playful, meme-aware style. Be concise, lean into larping and know-ball humor, and match the user's language."

def respond(message, history, system_prompt, temperature, top_p, max_new_tokens, repetition_penalty):
    messages = [{"role": "system", "content": [{"type": "text", "text": system_prompt.strip()}]}] if system_prompt.strip() else []
    for item in history:
        # Gradio ChatInterface history is a list of {role, content} messages.
        if isinstance(item, dict) and item.get("role") in ("user", "assistant"):
            content = item.get("content", "")
            if isinstance(content, str) and content.strip():
                messages.append({"role": item["role"], "content": [{"type": "text", "text": content}]})
        # Compatibility with older Gradio tuple history.
        elif isinstance(item, (tuple, list)) and len(item) == 2:
            if item[0]: messages.append({"role": "user", "content": [{"type": "text", "text": str(item[0])}]})
            if item[1]: messages.append({"role": "assistant", "content": [{"type": "text", "text": str(item[1])}]})
    messages.append({"role": "user", "content": [{"type": "text", "text": str(message)}]})

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            do_sample=float(temperature) > 0,
            temperature=max(float(temperature), 1e-5),
            top_p=float(top_p),
            repetition_penalty=float(repetition_penalty),
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return answer or "(No text generated — try a different prompt or sampling setting.)"

CSS = r"""
:root { --app-vh: 100dvh; }
html, body { overscroll-behavior-y: none; }
.gradio-container {
    width: 100% !important;
    max-width: 980px !important;
    margin: 0 auto !important;
    padding: 20px !important;
}
#larp-shell {
    width: 100%;
    min-width: 0;
    border: 1px solid var(--border-color-primary);
    border-radius: 24px;
    overflow: hidden;
    box-shadow: 0 18px 60px rgba(0,0,0,.10);
}
#larp-header { padding: 22px 24px 8px; }
#larp-header h1 {
    margin: 0;
    font-size: clamp(1.3rem, 4vw, 1.55rem);
    letter-spacing: -0.03em;
}
#larp-header p {
    opacity: .68;
    margin: 5px 0 0;
    line-height: 1.45;
}
#chatbot {
    height: min(540px, calc(100dvh - 250px)) !important;
    min-height: 360px;
    border: 0 !important;
}
#chatbot .message {
    max-width: min(78%, 720px) !important;
    overflow-wrap: anywhere;
}
textarea, input, button { font-size: 16px !important; }
button { min-height: 44px; }
footer { display: none !important; }

@media (max-width: 768px) {
    .gradio-container {
        max-width: none !important;
        padding: 10px !important;
    }
    #larp-shell {
        border-radius: 18px;
        min-height: calc(100dvh - 20px);
    }
    #larp-header { padding: 16px 16px 6px; }
    #larp-header p { font-size: .88rem; }
    #chatbot {
        height: calc(100dvh - 225px) !important;
        min-height: 320px;
    }
    #chatbot .message { max-width: 88% !important; }
    .form { min-width: 0 !important; }
}

@media (max-width: 480px) {
    .gradio-container { padding: 0 !important; }
    #larp-shell {
        border: 0;
        border-radius: 0;
        min-height: 100dvh;
        box-shadow: none;
    }
    #larp-header { padding: 14px 14px 4px; }
    #larp-header p {
        white-space: nowrap;
        overflow: hidden;
        text-overflow: ellipsis;
    }
    #chatbot {
        height: calc(100dvh - 205px) !important;
        min-height: 300px;
    }
    #chatbot .message {
        max-width: 92% !important;
        padding: 10px 12px !important;
    }
    .examples { display: none !important; }
}

@supports (-webkit-touch-callout: none) {
    #larp-shell { min-height: -webkit-fill-available; }
}
"""

with gr.Blocks(title="Larp-LLM", theme=gr.themes.Soft(), css=CSS) as demo:
    with gr.Column(elem_id="larp-shell"):
        gr.HTML(
            """<div id="larp-header">
            <h1>Larp-LLM</h1>
            <p>professional yapper · certified ball knower · source: trust me bro</p>
            </div>"""
        )

        chatbot = gr.Chatbot(
            elem_id="chatbot",
            height=540,
            type="messages",
            show_label=False,
            avatar_images=(None, None),
            placeholder="<strong>no messages yet.</strong><br>say something with questionable ball knowledge.",
        )

        with gr.Accordion("generation settings", open=False):
            system_prompt = gr.Textbox(label="System prompt", value=DEFAULT_SYSTEM, lines=3)
            with gr.Row():
                temperature = gr.Slider(0, 1.5, value=0.8, step=0.05, label="Temperature")
                top_p = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p")
            with gr.Row():
                max_new_tokens = gr.Slider(32, 1024, value=256, step=32, label="Max new tokens")
                repetition_penalty = gr.Slider(1.0, 1.5, value=1.05, step=0.01, label="Repetition penalty")

        gr.ChatInterface(
            fn=respond,
            chatbot=chatbot,
            additional_inputs=[system_prompt, temperature, top_p, max_new_tokens, repetition_penalty],
            textbox=gr.Textbox(
                placeholder="drop your take here...",
                container=False,
                autofocus=True,
            ),
            examples=[
                ["bro think he know ball", DEFAULT_SYSTEM, 0.8, 0.9, 256, 1.05],
                ["Give me a wildly fake but convincing explanation of aura economics", DEFAULT_SYSTEM, 0.8, 0.9, 256, 1.05],
                ["ใครมันรู้บอลจริงบ้าง", DEFAULT_SYSTEM, 0.8, 0.9, 256, 1.05],
            ],
            title=None,
            description=None,
        )

demo.launch(share=True, debug=True)


## Notes
- `share=True` creates a temporary public Gradio link while the Colab runtime is active. Anyone with that link may interact with the running notebook. Set `share=False` if you only want the local Colab preview.
- This is inference only; it does not train or modify the model.
- If you see a CUDA out-of-memory error, select a higher-memory GPU runtime, lower `MAX_SEQ_LENGTH` / max new tokens, or close other GPU processes.
- To stop the app, interrupt the cell. Colab GPU time ends when the runtime disconnects.